# 개별종목 조합I — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4953,0.5012,-0.0059,0.3244,0.3640,0.0772,0.3810,0.1113,0.2129
1,2,balanced,980,20150123,20150421,0.3871,0.3978,-0.0108,0.3479,0.3589,0.0495,0.3734,0.1939,0.2826
2,3,balanced,1210,20151228,20160328,0.3570,0.3762,-0.0192,0.3560,0.3561,0.0375,0.3681,0.3278,0.3464
3,4,balanced,1439,20161202,20170228,0.4515,0.4617,-0.0102,0.3603,0.3768,0.0866,0.3973,0.1619,0.2686
4,5,balanced,1669,20171113,20180207,0.4162,0.3901,0.0261,0.3815,0.3931,0.1001,0.3880,0.2587,0.3375
5,6,balanced,1899,20181024,20190118,0.4099,0.3725,0.0375,0.4088,0.4184,0.1300,0.4202,0.5185,0.4403
6,7,balanced,2129,20190930,20191224,0.4635,0.4781,-0.0147,0.3549,0.3752,0.0870,0.3981,0.1908,0.2936
7,8,balanced,2359,20200902,20201130,0.3942,0.3476,0.0466,0.3912,0.3951,0.0932,0.3973,0.4245,0.4028
8,9,balanced,2589,20210806,20211105,0.3976,0.3914,0.0062,0.3873,0.3949,0.0915,0.3876,0.3245,0.3668
9,10,balanced,2818,20220714,20221012,0.3382,0.3454,-0.0072,0.3384,0.3418,0.0144,0.3551,0.2676,0.3109


,OOS 폴드 평균
accuracy,0.4084
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0115
macro_f1,0.3696
balanced_accuracy,0.3801
mcc,0.0794
pr_auc_macro_ovr,0.3884
down_recall,0.2875
core_harmonic_mean,0.3338


재실행 명령: python scripts/run_stock_model_experiment.py
